# PWM Example 3 — TIM1 DMA sequence and advanced PWM

**Target:** STM32F103C8T6 · TIM1_CH4 / PA11  
**Focus:** Trace the repository's update-triggered DMA duty sequence before studying optional power-control features.

## Feature classification

| Variant | Timer support | Key behavior |
|---|---|---|
| Edge-aligned | TIM1–TIM4 | Counter counts one direction; `f = f_TIM / ((PSC+1)(ARR+1))` |
| Center-aligned | TIM1–TIM4 where channel mode supports it | Counter counts up and down; symmetric edges; frequency approximately `f_TIM / (2(PSC+1)ARR)` |
| Complementary output | TIM1 CH1N–CH3N | Inverted paired outputs with configurable dead time |
| Break input | TIM1 | Hardware fault input can force outputs to configured safe states |
| DMA duty sequence | TIM1 update request | Transfer successive compare values without per-period CPU writes |

## Repository example — `STM32F103C8T6_Basic_Timer_IT`

This project name is misleading: it configures TIM1_CH4 PWM and DMA1 Channel 5 on TIM1 update, not a basic timer interrupt. Its clock is 72 MHz, PSC=71 and ARR=99, so the edge-aligned carrier is 10 kHz. A three-element CCR4 buffer `{75, 50, 25}` sequences approximately 75%, 50%, and 25% compare intervals each update. The output polarity is low, so reason about the physical active-low waveform rather than assuming active-high duty. CH4 is PA11, which may conflict with USB D−.

The project configures dead time as zero and disables the break input. It is a DMA-learning example only, not a motor/bridge safety reference. Trace `MX_DMA_Init`, TIM1 update DMA enable, DMA buffer values, and `HAL_TIM_PWM_Start` in CubeIDE.

Center-aligned frequency formula depends on exact counting and update configuration; verify against RM0008 and CubeMX settings. The simpler edge-aligned formula from the first PWM workshop does not apply unchanged.

## Center-aligned setup

Select center-aligned counter mode in CubeMX and calculate the period for the up/down traversal. Verify which update events fire and how preload values transfer. Center alignment often reduces switching harmonics but changes update frequency and event timing.

## Optional extension — center-aligned / complementary output

TIM1 is the F103C8T6 advanced-control timer. Default main outputs PA8/PA9/PA10 correspond to CH1/CH2/CH3; complementary outputs are PB13/PB14/PB15. TIM1_CH4 on PA11 has no CH4N output. Configure PWM on the main channel, output polarity and idle state, complementary output polarity/state, dead-time generator, and break input. The main output enable (MOE) and HAL API must be handled for outputs to reach the pins.

The current `STM32F103C8T6_Basic_Timer_IT` project does not enable complementary output. To explore it, create a separate CubeMX variation using CH1–CH3 and their N outputs; do not repurpose its CH4 setup. HAL starts main and complementary outputs separately, for example `HAL_TIM_PWM_Start(&htim1, TIM_CHANNEL_1)` and `HAL_TIMEx_PWMN_Start(&htim1, TIM_CHANNEL_1)`. Follow generated initialization and verify advanced-timer break/MOE behavior before testing.

```c
// Only after TIM1 is configured for PWM + complementary output + dead time.
HAL_TIM_PWM_Start(&htim1, TIM_CHANNEL_1);
HAL_TIMEx_PWMN_Start(&htim1, TIM_CHANNEL_1);
```


> **Safety warning:** Complementary PWM is intended for external gate-driver/power circuitry, not direct connection between MCU pins and a motor/bridge. Choose dead time from device/driver switching characteristics, define break fault polarity and safe output state, and validate with a scope at low voltage/current. Never assume a software reset alone guarantees a safe bridge state.

## Checklist

1. Confirm TIM1 and exact pin mapping/remap in CubeMX and the MCU datasheet.
2. Calculate timer clock from APB2, then derive period and duty with alignment-specific formula.
3. Set preload, polarity, idle states, repetition count, dead time, and break behavior intentionally.
4. Check external driver, current/voltage limits, bootstrap requirements, and fault response before applying power.
5. Verify waveform and complementary non-overlap using a scope or logic analyzer.